# Machine Learning PAM50 Classification of Multi-Omics Breast Cancer Data

## Data Preprocessing - TCGA

In [65]:
import pandas as pd
from collections import Counter

# Base folder where all TCGA .csv files are stored
base = "tcga-data/"

# -----------------------------
# Clinical sample data
# Contains PAM50 molecular subtype labels
# -----------------------------
clinical = pd.read_csv(
    base + "54814622_BRCA_label_num.csv"
)

# -----------------------------
# mRNA expression data (z-score normalized)
# Rows = genes, columns = samples
# Used to extract PAM50 gene expression features
# -----------------------------

expr = pd.read_csv(
    base + "BRCA_mRNA_aligned.csv",
    index_col=0
)

# -----------------------------
# Copy Number Alteration (CNA) data
# Values typically: -2, -1, 0, 1, 2
# Used for ERBB2, TP53, MYC, etc.
# -----------------------------

cnv = pd.read_csv(
    base + "BRCA_CNV_aligned.csv",
    index_col=0
)

# -----------------------------
# Methylation data  (z-score normalized)
# -----------------------------

methyl = pd.read_csv(
    base + "BRCA_Methy_aligned.csv",
    index_col=0
)

# -----------------------------
# miRNA data  (z-score normalized)
# -----------------------------

mirna = pd.read_csv(
    base + "BRCA_miRNA_aligned.csv",
    index_col=0
)

In [66]:
print(clinical.head())
print(expr.head())
print(cnv.head())
print(methyl.head())
print(mirna.head())

   Label
0      0
1      1
2      2
3      0
4      0
              TCGA.3C.AAAU.01  TCGA.3C.AALI.01  TCGA.3C.AALJ.01  \
FBXL15               0.806517         0.008469        -1.529268   
LOC105372824         0.658390        -1.347314        -0.357091   
TMEM41B              0.012981        -0.051182         0.084653   
CHUK                 0.180961        -0.304748        -2.157297   
ZNF518B              0.243034         0.856080         0.700889   

              TCGA.3C.AALK.01  TCGA.5L.AAT0.01  TCGA.5T.A9QA.01  \
FBXL15               0.387512         0.539446         0.062674   
LOC105372824        -0.780478         0.407596         0.822078   
TMEM41B             -0.565627         0.423900         1.979520   
CHUK                -0.319247        -0.535740         0.573746   
ZNF518B              0.475732         1.006535        -1.861611   

              TCGA.A1.A0SB.01  TCGA.A1.A0SE.01  TCGA.A1.A0SF.01  \
FBXL15              -1.585499        -0.663846         1.076331   
LOC105

In [67]:
# PAM50 gene list
pam50_genes = [
    "ACTR3B","BAG1","BCL2","CCNB1","CDH3","CENPF","CEP55",
    "EXO1","FOXC1","GPR160","GRB7","KRT5","KRT14","KRT17",
    "MKI67","MLPH","NDC80","NKI","NME1","NUF2","PGR","PHGDH",
    "PTTG1","RRM2","SFRP1","SLC39A6","TMEM45B","TYMS","UBE2C",
    "UBE2T","VIM","ANLN","CCNE1","CDC20","CDC6","CXXC5",
    "EGFR","ERBB2","ESR1","GPR160","KIF2C","KRT5","MMP11",
    "ORC6L","PIM1","RRM2","SFRP1","TPX2","UBE2C","MYBL2"
]

# Remove duplicates just in case
pam50_genes = list(set(pam50_genes))

# PAM50 genes uppercase
pam50_genes = [g.upper() for g in pam50_genes]

# Make all gene names uppercase in datasets
expr.index = [g.upper() for g in expr.index]
cnv.index = [g.upper() for g in cnv.index]
methyl.index = [g.upper() for g in methyl.index]

In [68]:
# Add Sample_ID column to clinical df
expr_for_samples = expr.T
samples_list = expr_for_samples.index.tolist()
clinical['Sample_ID'] = samples_list
clinical = clinical.set_index('Sample_ID')

print(clinical.head())

                 Label
Sample_ID             
TCGA.3C.AAAU.01      0
TCGA.3C.AALI.01      1
TCGA.3C.AALJ.01      2
TCGA.3C.AALK.01      0
TCGA.5L.AAT0.01      0


In [69]:
# Expression and CNV
expr_pam50 = expr.loc[expr.index.isin(pam50_genes)]
expr_pam50 = expr_pam50.T
print(expr_pam50.head())

cnv_pam50 = cnv.loc[cnv.index.isin(pam50_genes)]
cnv_pam50 = cnv_pam50.T
print(cnv_pam50.head())

# Methylation: average if multiple probes per gene
# Filter to PAM50 genes
methyl_pam50 = methyl.loc[methyl.index.isin(pam50_genes)]

# If multiple probes per gene, take the mean per gene
methyl_pam50 = methyl_pam50.groupby(methyl_pam50.index).mean()

# Transpose so rows = samples, columns = features
methyl_pam50 = methyl_pam50.T
print(methyl_pam50.head())

# miRNA: transpose so rows = samples, columns = miRNAs
mirna_pam50 = mirna.T  # if columns are samples, transpose to align
print(mirna_pam50.head())

                     TYMS     MKI67       VIM      ESR1      NUF2     PHGDH  \
TCGA.3C.AAAU.01  0.086297  0.776142 -1.498302 -0.039178  0.278465 -0.156844   
TCGA.3C.AALI.01  0.503223  0.591314 -0.363568 -1.710956  0.832529 -0.169426   
TCGA.3C.AALJ.01  0.390210 -0.385621  0.429284  0.430437  0.765321 -1.538990   
TCGA.3C.AALK.01  0.284101 -0.264438  0.206574 -0.322726 -0.493342 -0.303171   
TCGA.5L.AAT0.01 -0.767300 -0.982322  0.543472  0.349409 -1.219799 -0.558496   

                     BCL2     KRT14     CDC20     CENPF  ...   TMEM45B  \
TCGA.3C.AAAU.01  0.977815  0.076081  0.126100  0.254780  ... -1.263301   
TCGA.3C.AALI.01 -1.402084 -0.569699  0.161995  0.441229  ...  1.229669   
TCGA.3C.AALJ.01  1.018905 -1.575649  0.491063  0.078986  ... -1.218088   
TCGA.3C.AALK.01  0.752807  0.701028 -0.130623 -0.001543  ...  0.790900   
TCGA.5L.AAT0.01  0.809507  0.506460 -0.989186 -0.782230  ...  0.520426   

                     MLPH     UBE2T      RRM2     PTTG1     CCNB1     KRT17  \
T

In [70]:
# Find common samples across all dataframes
samples = set(clinical.index)
samples &= set(expr_pam50.index)
samples &= set(cnv_pam50.index)
samples &= set(methyl_pam50.index)
samples &= set(mirna_pam50.index)
samples = sorted(list(samples))

# Subset all dataframes
expr_pam50 = expr_pam50.loc[samples]
cnv_pam50 = cnv_pam50.loc[samples]
methyl_pam50 = methyl_pam50.loc[samples]
mirna_pam50 = mirna_pam50.loc[samples]

In [71]:
print(clinical.head())
print(expr_pam50.head())
print(cnv_pam50.head())
print(methyl_pam50.head())
print(mirna_pam50.head())

                 Label
Sample_ID             
TCGA.3C.AAAU.01      0
TCGA.3C.AALI.01      1
TCGA.3C.AALJ.01      2
TCGA.3C.AALK.01      0
TCGA.5L.AAT0.01      0
                     TYMS     MKI67       VIM      ESR1      NUF2     PHGDH  \
TCGA.3C.AAAU.01  0.086297  0.776142 -1.498302 -0.039178  0.278465 -0.156844   
TCGA.3C.AALI.01  0.503223  0.591314 -0.363568 -1.710956  0.832529 -0.169426   
TCGA.3C.AALJ.01  0.390210 -0.385621  0.429284  0.430437  0.765321 -1.538990   
TCGA.3C.AALK.01  0.284101 -0.264438  0.206574 -0.322726 -0.493342 -0.303171   
TCGA.5L.AAT0.01 -0.767300 -0.982322  0.543472  0.349409 -1.219799 -0.558496   

                     BCL2     KRT14     CDC20     CENPF  ...   TMEM45B  \
TCGA.3C.AAAU.01  0.977815  0.076081  0.126100  0.254780  ... -1.263301   
TCGA.3C.AALI.01 -1.402084 -0.569699  0.161995  0.441229  ...  1.229669   
TCGA.3C.AALJ.01  1.018905 -1.575649  0.491063  0.078986  ... -1.218088   
TCGA.3C.AALK.01  0.752807  0.701028 -0.130623 -0.001543  ...  0.7909

In [72]:
print(clinical.index[:5])
print(expr_pam50.index[:5])
print(cnv_pam50.index[:5])
print(methyl_pam50.index[:5])
print(mirna_pam50.index[:5])

Index(['TCGA.3C.AAAU.01', 'TCGA.3C.AALI.01', 'TCGA.3C.AALJ.01',
       'TCGA.3C.AALK.01', 'TCGA.5L.AAT0.01'],
      dtype='object', name='Sample_ID')
Index(['TCGA.3C.AAAU.01', 'TCGA.3C.AALI.01', 'TCGA.3C.AALJ.01',
       'TCGA.3C.AALK.01', 'TCGA.5L.AAT0.01'],
      dtype='object')
Index(['TCGA.3C.AAAU.01', 'TCGA.3C.AALI.01', 'TCGA.3C.AALJ.01',
       'TCGA.3C.AALK.01', 'TCGA.5L.AAT0.01'],
      dtype='object')
Index(['TCGA.3C.AAAU.01', 'TCGA.3C.AALI.01', 'TCGA.3C.AALJ.01',
       'TCGA.3C.AALK.01', 'TCGA.5L.AAT0.01'],
      dtype='object')
Index(['TCGA.3C.AAAU.01', 'TCGA.3C.AALI.01', 'TCGA.3C.AALJ.01',
       'TCGA.3C.AALK.01', 'TCGA.5L.AAT0.01'],
      dtype='object')


In [73]:
# Add feature tags
expr_pam50.columns = [f"{gene}_expr" for gene in expr_pam50.columns]
cnv_pam50.columns = [f"{gene}_cnv" for gene in cnv_pam50.columns]
methyl_pam50.columns = [f"{gene}_methyl" for gene in methyl_pam50.columns]
mirna_pam50.columns = [f"{miRNA}_mirna" for miRNA in mirna_pam50.columns]

# Concatenate all features
df = pd.concat([expr_pam50, cnv_pam50, methyl_pam50, mirna_pam50], axis=1)

# Add PAM50 column (integer labels)
df['PAM50'] = clinical.loc[df.index, 'Label']

# Remove any columns named 'nan_mirna'
df = df.loc[:, df.columns != 'nan_mirna']

# Add PAM50 string label column
pam50_map = {0: 'LumA', 1: 'LumB', 2: 'Her2', 3: 'Basal', 4: 'Normal'}
df['PAM50_Label'] = df['PAM50'].map(pam50_map)

print(df.shape)
print(df.head())

# Export .csv
df.to_csv("tcga_data.csv", index=True)

(671, 360)
                 TYMS_expr  MKI67_expr  VIM_expr  ESR1_expr  NUF2_expr  \
TCGA.3C.AAAU.01   0.086297    0.776142 -1.498302  -0.039178   0.278465   
TCGA.3C.AALI.01   0.503223    0.591314 -0.363568  -1.710956   0.832529   
TCGA.3C.AALJ.01   0.390210   -0.385621  0.429284   0.430437   0.765321   
TCGA.3C.AALK.01   0.284101   -0.264438  0.206574  -0.322726  -0.493342   
TCGA.5L.AAT0.01  -0.767300   -0.982322  0.543472   0.349409  -1.219799   

                 PHGDH_expr  BCL2_expr  KRT14_expr  CDC20_expr  CENPF_expr  \
TCGA.3C.AAAU.01   -0.156844   0.977815    0.076081    0.126100    0.254780   
TCGA.3C.AALI.01   -0.169426  -1.402084   -0.569699    0.161995    0.441229   
TCGA.3C.AALJ.01   -1.538990   1.018905   -1.575649    0.491063    0.078986   
TCGA.3C.AALK.01   -0.303171   0.752807    0.701028   -0.130623   -0.001543   
TCGA.5L.AAT0.01   -0.558496   0.809507    0.506460   -0.989186   -0.782230   

                 ...  hsa.mir.6837_mirna  hsa.mir.6842_mirna  \
TCGA.3C.AAA